# C01 — Medidas, numerario, Black-76 y Bachelier

**Objetivos:** comprobar con simulación por qué las fórmulas de mercado funcionan (cambio de numerario); comparar
Bachelier, Black-76 y Black desplazado; ajustes de convexidad (*in arrears*, CMS); Greeks de opciones de tipos;
caso de las dos cotizaciones del mismo cap.

**Fuentes:** Hull cap. 28, 29 y 30; Veronesi cap. 20 y 21. Datos sintéticos.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> repository root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

from derivlab.market import sample_market
from derivlab.curves import build_curves, discount_t, shift_curve
from derivlab.dates import spot_date, add_months
from derivlab.scenarios import HullWhite, pathwise_discount, ACT360
from derivlab.rate_options import (bachelier, black76, shifted_black, implied_normal_vol, caplets, cap_value,
                                   normal_vol_from_shifted_black, bachelier_vega)
from derivlab.convexity import cms_convexity_adjustment, in_arrears_adjustment
from derivlab.short_rate import forward_swap_today, swap_legs_at, hw_implied_normal_vol

m = sample_market()
estr, eur = build_curves(m)
ref, spot = m["ref_date"], spot_date(m["ref_date"])
hw = HullWhite(estr, eur, a=0.03, sigma=0.0085)

## 1. Cambio de numerario en una simulación

Euribor 6M que fija en T₁ = 4,5 años y paga en T₂ = 5 años. Bajo la medida neutral al riesgo (numerario: cuenta
bancaria) el Euribor esperado no es el forward. Bajo la medida forward T₂ (numerario: el bono P(t,T₂)) sí lo es:

$$\mathbb{E}^{Q}\left[D(0,T_2)\,L(T_1)\right] = P(0,T_2)\,\mathbb{E}^{T_2}[L(T_1)] = P(0,T_2)\,F(0;T_1,T_2)$$

Esa igualdad es la que justifica valorar un caplet con el forward y descontar con P(0, T₂).

In [2]:
T1, T2 = 4.5, 5.0
times = np.array([0.0, T1, T2])
x = hw.simulate(times, 200000, seed=11)
D = pathwise_discount(estr, hw.a, hw.sigma, times, x)
L = hw.euribor_fixing(T1, x[:, 1])
tau = 0.5 * ACT360
F = (discount_t(eur, T1) / discount_t(eur, T2) - 1) / tau
naive = L.mean()
fwd_measure = (D[:, 2] * L).mean() / discount_t(estr, T2)
print(f"CASE forward F = {F:.4%}; E^Q[L] = {naive:.4%} (diff {(naive - F) * 1e4:+.1f} bp); "
      f"E^Q[D(0,T2) L]/P(0,T2) = {fwd_measure:.4%} (diff {(fwd_measure - F) * 1e4:+.2f} bp)")

K = F
mc_caplet = (D[:, 2] * np.maximum(L - K, 0)).mean() * tau
vol_impl = implied_normal_vol(mc_caplet / (tau * discount_t(estr, T2)), F, K, T1)
print(f"CASE ATM caplet: Monte Carlo {mc_caplet * 1e4:.2f} bp of notional; implied normal vol {vol_impl * 1e4:.1f} bp")

CASE forward F = 2.6807%; E^Q[L] = 2.7583% (diff +7.8 bp); E^Q[D(0,T2) L]/P(0,T2) = 2.6797% (diff -0.10 bp)
CASE ATM caplet: Monte Carlo 30.27 bp of notional; implied normal vol 78.9 bp


## 2. Tres fórmulas para el mismo caplet

- **Bachelier (normal):** el tipo se mueve en puntos básicos; admite tipos negativos. Estándar en EUR.
- **Black-76 (lognormal):** el tipo se mueve en porcentaje; no admite F ≤ 0.
- **Black desplazado:** Black sobre F + s y K + s; con s = 1 % o 2 % admite tipos negativos.

Regla de conversión cerca del dinero: σ_N ≈ σ_LN · (F + s).

In [3]:
from scipy.optimize import brentq
t, Fc = 3.0, 0.026
p_atm = float(bachelier(Fc, Fc, t, 0.0085))
v_ln = brentq(lambda v: black76(Fc, Fc, t, v) - p_atm, 0.01, 2.0)                  # same ATM price in Black
v_sh = brentq(lambda v: shifted_black(Fc, Fc, t, v, 0.01) - p_atm, 0.01, 2.0)      # same ATM price, 1% shift
rows = []
for k in [0.015, 0.020, 0.026, 0.030, 0.035, 0.040]:
    p_norm = float(bachelier(Fc, k, t, 0.0085))
    rows.append({"strike": k, "bachelier_bp": p_norm * 1e4, "black_bp": black76(Fc, k, t, v_ln) * 1e4,
                 "shifted_black_bp": shifted_black(Fc, k, t, v_sh, 0.01) * 1e4})
fm = pd.DataFrame(rows)
print(f"Vol equivalents at the money: normal 85 bp = lognormal {v_ln:.1%} = shifted (1%) {v_sh:.1%} "
      f"(rule of thumb: {0.0085 / Fc:.1%} and {0.0085 / (Fc + 0.01):.1%})")
print(fm.to_string(index=False, formatters={"strike": "{:.2%}".format, "bachelier_bp": "{:.2f}".format,
                                             "black_bp": "{:.2f}".format, "shifted_black_bp": "{:.2f}".format}))
print("Negative forward -0.30%, strike 0%: Black =", black76(-0.003, 0.0, 1.0, 0.3),
      f"; Bachelier = {float(bachelier(-0.003, 0.0, 1.0, 0.006)) * 1e4:.2f} bp")

Vol equivalents at the money: normal 85 bp = lognormal 33.1% = shifted (1%) 23.8% (rule of thumb: 32.7% and 23.6%)
strike bachelier_bp black_bp shifted_black_bp
 1.50%       129.41   119.96           122.58
 2.00%        93.55    87.11            88.94
 2.60%        58.73    58.73            58.73
 3.00%        40.89    45.15            43.98
 3.50%        24.38    32.61            30.36
 4.00%        13.46    23.69            20.83
Negative forward -0.30%, strike 0%: Black = nan ; Bachelier = 11.87 bp


Las tres coinciden en el dinero por construcción y se separan en las alas: cada fórmula implica un *smile*
distinto. Por eso una volatilidad no se puede comparar entre bancos sin saber en qué modelo se cotiza.

## 3. Caso: dos cotizaciones del mismo cap

Un tesorero pide precio para un cap a 5 años al 3,00 % sobre 100 M€ (Euribor 6M):

- **Banco A:** 90 pb de volatilidad normal, plana.
- **Banco B:** 25 % lognormal desplazada (desplazamiento del 1 %), plana.

¿Cuál es más barato? Se valoran ambos caplet a caplet y se traduce la cotización de B a volatilidad normal en
cada caplet.

In [4]:
N, K_cap, end5 = 100e6, 0.030, add_months(spot, 60)
cl = caplets(ref, estr, eur, spot, end5, N, K_cap, 0.0090)
price_a = sum(c["pv"] for c in cl)
rows = []
price_b = 0.0
for c in cl:
    pb = N * c["tau"] * c["df"] * shifted_black(c["forward"], K_cap, c["t_fix"], 0.25, 0.01)
    price_b += pb
    rows.append({"t_fix": round(c["t_fix"], 2), "forward": c["forward"],
                 "normal_equiv_B_bp": normal_vol_from_shifted_black(c["forward"], K_cap, c["t_fix"], 0.25, 0.01) * 1e4})
eq = pd.DataFrame(rows)
print(eq.iloc[::2].to_string(index=False, formatters={"forward": "{:.3%}".format, "normal_equiv_B_bp": "{:.1f}".format}))
cheaper = "A" if price_a < price_b else "B"
print(f"CASE cap 5y 3.00% on 100M: bank A {price_a:,.0f} EUR, bank B {price_b:,.0f} EUR; cheaper: bank {cheaper} "
      f"by {abs(price_a - price_b):,.0f} EUR")
print(f"CASE bank B's 25% shifted vol is worth {eq.normal_equiv_B_bp.min():.0f}-{eq.normal_equiv_B_bp.max():.0f} bp normal across caplets")

 t_fix forward normal_equiv_B_bp
   0.5  2.028%              87.2
   1.5  2.236%              89.8
   2.5  2.381%              91.5
   3.5  2.543%              93.3
   4.5  2.681%              94.8
CASE cap 5y 3.00% on 100M: bank A 1,351,010 EUR, bank B 1,413,203 EUR; cheaper: bank A by 62,193 EUR
CASE bank B's 25% shifted vol is worth 87-95 bp normal across caplets


### Greeks del cap del banco A: delta en DV01 y vega por punto básico de volatilidad

In [5]:
up = sum(c["pv"] for c in caplets(ref, shift_curve(estr, 1), shift_curve(eur, 1), spot, end5, N, K_cap, 0.0090))
dv01 = up - price_a
vega = sum(c["pv"] for c in caplets(ref, estr, eur, spot, end5, N, K_cap, 0.0091)) - price_a
swap_dv01 = 100e6 * sum(c["tau"] * c["df"] for c in cl) * 1e-4
print(f"CASE cap delta {dv01:,.0f} EUR/bp ({dv01 / swap_dv01:.0%} of a 5y swap's DV01); vega {vega:,.0f} EUR per bp of vol")

CASE cap delta 12,507 EUR/bp (29% of a 5y swap's DV01); vega 22,556 EUR per bp of vol


## 4. Ajustes de convexidad

**Euribor *in arrears*.** Si el Euribor que fija en T₁ se paga en T₁ en lugar de en T₂, su valor esperado bajo
la medida T₁ es mayor que el forward (Hull ec. 30.2, versión normal): $F + \sigma_N^2 T_1 \tau/(1 + F\tau)$.

**CMS.** Un pago en T igual al swap a 10 años que fija en T vale más que el swap forward (Hull ec. 30.1):
$S_0 - \tfrac12 \sigma_N^2 T\, G''(S_0)/G'(S_0)$.

In [6]:
sig_cap = vol_impl
in_arrears_mc = (D[:, 1] * L).mean() / discount_t(estr, T1)
in_arrears_formula = F + in_arrears_adjustment(F, sig_cap, T1, tau)
print(f"CASE in arrears: forward {F:.4%}; Monte Carlo {in_arrears_mc:.4%}; formula {in_arrears_formula:.4%} "
      f"(adjustment {(in_arrears_formula - F) * 1e4:.2f} bp)")

T = 5.0
tt = np.array([0.0, T])
x5 = hw.simulate(tt, 200000, seed=12)
D5 = pathwise_discount(estr, hw.a, hw.sigma, tt, x5)[:, 1]
S0, A0 = forward_swap_today(hw, T, 10)
A5, F5 = swap_legs_at(hw, T, T, 10, x5[:, 1])
S5 = F5 / A5
cms_mc = (D5 * S5).mean() / discount_t(estr, T)
sig_sw = hw_implied_normal_vol(hw, T, 10)
cms_formula = S0 + cms_convexity_adjustment(S0, sig_sw, T, 10)
print(f"CASE CMS 10y in 5y: forward swap {S0:.4%}; Monte Carlo {cms_mc:.4%} (+{(cms_mc - S0) * 1e4:.1f} bp); "
      f"Hull 30.1 formula {cms_formula:.4%} (+{(cms_formula - S0) * 1e4:.1f} bp)")

CASE in arrears: forward 2.6807%; Monte Carlo 2.6938%; formula 2.6947% (adjustment 1.40 bp)


CASE CMS 10y in 5y: forward swap 3.0980%; Monte Carlo 3.2292% (+13.1 bp); Hull 30.1 formula 3.2247% (+12.7 bp)


La fórmula de Hull se queda algo corta frente a la simulación: aproxima el swap como el rendimiento de un bono con
cupón fijo y no incluye que, en un modelo con reversión a la media, la volatilidad del swap y la del descuento
no son las mismas. En la práctica el CMS se valora replicándolo con swaptions de todos los strikes, que recoge
además el *smile*.

## 5. Comprobaciones

In [7]:
assert abs(fwd_measure - F) < 0.00005                 # forward is a martingale under the T2-forward measure
assert abs(naive - F) > abs(fwd_measure - F)
assert fm.loc[fm.strike == 0.026, ["bachelier_bp", "black_bp", "shifted_black_bp"]].std(axis=1).iloc[0] < 1e-8
assert abs(in_arrears_mc - in_arrears_formula) < 0.0001
assert 0 < cms_formula - S0 < cms_mc - S0 + 0.0005
assert 0 < dv01 < swap_dv01
print("All checks passed")

All checks passed


## Ejercicios

1. **Básico.** Convierte 30 % lognormal con forward 2,5 % a volatilidad normal (sin desplazamiento y con 2 %).
2. **Intermedio.** ¿A qué volatilidad normal plana el banco A cotizaría lo mismo que el banco B?
3. **Avanzado.** Calcula el ajuste CMS del swap a 10 años observado en 2, 5 y 10 años con la fórmula de Hull. ¿Cómo crece con el plazo?

## Soluciones

In [8]:
print(f"No shift: {0.30 * 0.025 * 1e4:.0f} bp; shifted 2%: {0.30 * (0.025 + 0.02) * 1e4:.0f} bp (ATM approximation)")

No shift: 75 bp; shifted 2%: 135 bp (ATM approximation)


In [9]:
from scipy.optimize import brentq
flat = brentq(lambda v: cap_value(ref, estr, eur, spot, end5, N, K_cap, v) - price_b, 0.001, 0.03)
print(f"Flat normal vol equivalent to bank B: {flat * 1e4:.1f} bp")

Flat normal vol equivalent to bank B: 92.8 bp


In [10]:
for T_ in [2.0, 5.0, 10.0]:
    s0, _ = forward_swap_today(hw, T_, 10)
    adj = cms_convexity_adjustment(s0, hw_implied_normal_vol(hw, T_, 10), T_, 10)
    print(f"T = {T_:>4}: forward {s0:.3%}, CMS adjustment {adj * 1e4:.1f} bp")
print("Roughly linear in T for a given vol: the adjustment is sigma^2 x T x (G''/G').")

T =  2.0: forward 2.894%, CMS adjustment 5.5 bp
T =  5.0: forward 3.098%, CMS adjustment 12.7 bp
T = 10.0: forward 3.228%, CMS adjustment 22.1 bp
Roughly linear in T for a given vol: the adjustment is sigma^2 x T x (G''/G').
